# Практика 3. Graph Neural Network своими руками

Сегодня:
1. пишем слой GCN через scatter-операции и **сверяем с `GCNConv` из PyG** до машинной точности;
2. обучаем модель на Cora с ранней остановкой по валидации;
3. сравниваем с MLP, который графа не видит;
4. воспроизводим over-smoothing: делаем сеть глубже и смотрим, как всё разваливается.

Всё, что делаем сегодня, — по лекции 3. Сэмплирование соседей и большие графы будут в лекции 5.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

try:
    import torch_geometric  # noqa: F401
except ImportError:
    !pip install -q torch_geometric

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F          # функции без весов: relu, dropout, cross_entropy, normalize
from torch import nn                     # слои с обучаемыми весами: Linear, ModuleList
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv   # эталонный слой — с ним сверим свою реализацию
from torch_geometric.utils import add_self_loops, degree, to_undirected

# фиксируем случайность: начальные веса и dropout будут одинаковыми при каждом перезапуске
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device)

In [ ]:
dataset = Planetoid(root="/tmp/Cora", name="Cora")   # при первом запуске скачается
data = dataset[0]                     # в Cora один граф, поэтому берём его по индексу 0

# рёбра храним в обе стороны: сообщение должно идти и u → v, и v → u.
# В Cora из PyG они уже симметричны, to_undirected — дешёвая страховка на случай другого датасета
data.edge_index = to_undirected(data.edge_index)
data = data.to(device)                # переносим все тензоры графа на GPU, если он есть
print(data)

# маски — булевы векторы длины 2708: True у вершин, входящих в обучение, валидацию или тест.
# Модель видит ВЕСЬ граф, но учится только на метках train_mask
print(f"размечено: train {int(data.train_mask.sum())}, val {int(data.val_mask.sum())}, "
      f"test {int(data.test_mask.sum())}")

## 1. Слой GCN с нуля

Формула из лекции:

$$h_v' = \sigma\Big( W \sum_{u \in \mathcal{N}(v) \cup \{v\}} \frac{h_u}{\sqrt{d_u d_v}} \Big)$$

Три шага: добавить петли, посчитать нормировку, разослать и собрать сообщения.

Разберём, что за функции нам понадобятся:

- `add_self_loops(edge_index, num_nodes=n)` — дописывает к рёбрам петли `v → v`;
- `degree(col, n)` — считает, сколько раз каждая вершина встретилась как получатель;
- `h[row]` — собирает строки матрицы по списку индексов;
- `out.index_add_(0, col, msg)` — складывает строки `msg` в `out` по индексам `col`.

Всё остальное — ваша формула.

In [ ]:
class MyGCNConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # обучаемая матрица W из формулы. bias=False — чтобы точно совпасть с эталоном,
        # который в разделе 2 мы тоже создадим без bias
        self.lin = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, edge_index):
        # x: [n, in_dim] — признаки вершин; edge_index: [2, E] — рёбра столбцами (откуда, куда)
        n = x.size(0)

        # техническая часть — уже готова
        # петли v → v: вершина получает и своё собственное сообщение (∪ {v} в формуле).
        # ⚠ без петель вершина «забывает» свои признаки — первая типичная ошибка из лекции
        edge_index, _ = add_self_loops(edge_index, num_nodes=n)
        row, col = edge_index                       # row — откуда, col — куда; длина E + n
        # ⚠ степень считаем ПОСЛЕ петель: у вершины с двумя соседями d = 3, а не 2.
        # dtype=x.dtype — степени должны быть float, дальше из них берём корень.
        # clamp(min=1) — страховка от деления на ноль; с петлями степень и так не меньше 1
        deg = degree(col, n, dtype=x.dtype).clamp(min=1)
        # W применяем к вершинам ДО рассылки по рёбрам: n умножений вместо E, результат тот же
        h = self.lin(x)                             # [n, out_dim]

        # TODO 1: коэффициент для каждого ребра: 1 / sqrt(d_row * d_col)
        # подсказка: deg[row] и deg[col] — векторы длины «число рёбер»;
        # здесь вы берёте степень отправителя и получателя для каждого ребра
        norm = ...                                  # должно получиться [E + n]

        # TODO 2: сообщение по каждому ребру — признаки отправителя, взвешенные коэффициентом
        # подсказка: h[row] имеет форму [рёбра, признаки], norm — [рёбра],
        # поэтому norm нужно превратить в столбец: norm.unsqueeze(-1)
        msg = ...                                   # должно получиться [E + n, out_dim]

        # TODO 3: сложите сообщения в вершины-получатели
        # ⚠ out — по строке на ВЕРШИНУ, а не на ребро: поэтому zeros_like(h), а не zeros_like(msg).
        # ⚠ складываем по col (получатель), а не по row: иначе вершина «отдаст», а не «получит»
        out = torch.zeros_like(h)                   # [n, out_dim]
        ...
        return out                                  # σ здесь нет: ReLU поставим в модели между слоями

layer = MyGCNConv(dataset.num_features, 16).to(device)
out = layer(data.x, data.edge_index)
print("выход слоя:", out.shape)                     # ожидаем [2708, 16]

## 2. Сверка с PyG

Это самая важная ячейка практики. Копируем веса из нашего слоя в `GCNConv` и сравниваем выходы.
Расхождение должно быть на уровне `1e-6`, а не «примерно похоже».

In [ ]:
# эталон с теми же настройками, что у нас: петли, симметричная нормировка, без bias
ref = GCNConv(dataset.num_features, 16, bias=False, add_self_loops=True, normalize=True).to(device)

# копируем НАШИ веса в эталон: иначе оба слоя инициализированы случайно по-разному
# и выходы сравнивать бессмысленно. no_grad — копирование не должно попасть в граф вычислений
with torch.no_grad():
    ref.lin.weight.copy_(layer.lin.weight)      # у обоих Linear вес формы [out_dim, in_dim]

with torch.no_grad():
    mine = layer(data.x, data.edge_index)
    theirs = ref(data.x, data.edge_index)

# худшее расхождение по всем вершинам и признакам
diff = (mine - theirs).abs().max().item()
print(f"максимальное расхождение: {diff:.3e}")
# порог 1e-5, а не ноль: float32 складывает слагаемые в другом порядке, отсюда разница ~1e-7.
# Ошибка в формуле даёт расхождение порядка 1e-1 — спутать невозможно
assert diff < 1e-5, "слой реализован неверно — ищите ошибку в нормировке или петлях"
print("реализация совпадает с PyG")

Если расхождение большое, проверьте по порядку:
1. добавлены ли петли (`add_self_loops`);
2. считаются ли степени **после** добавления петель;
3. нормировка именно `d_row^(-1/2) * d_col^(-1/2)`, а не деление на одну степень;
4. агрегация по `col`, а не по `row`.

## 3. Собираем модель и обучаем

In [ ]:
class GNN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, layers=2, conv=MyGCNConv, dropout=0.5):
        super().__init__()
        # размеры слоёв: вход → скрытые → классы; при layers=2 это [1433, 64, 7]
        dims = [in_dim] + [hidden] * (layers - 1) + [out_dim]
        # ⚠ ModuleList, а не обычный list: иначе веса слоёв не попадут в model.parameters()
        # и оптимизатор их просто не увидит
        self.convs = nn.ModuleList([conv(dims[i], dims[i + 1]) for i in range(layers)])
        self.dropout = dropout

    def forward(self, x, edge_index, return_hidden=False):
        hidden = None
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:      # нелинейность и dropout — только МЕЖДУ слоями
                x = F.relu(x)                # σ из формулы: внутри слоя её нет, ставим здесь
                hidden = x                   # последний скрытый слой — понадобится в разделе 5
                # training=self.training: dropout работает при обучении и сам выключается в eval()
                x = F.dropout(x, p=self.dropout, training=self.training)
        # последний слой отдаёт логиты без ReLU: softmax сделает cross_entropy внутри себя
        return (x, hidden) if return_hidden else x


class MLP(nn.Module):
    """Тот же размер, но графа не видит — наш обязательный baseline."""
    def __init__(self, in_dim, hidden, out_dim, layers=2, dropout=0.5):
        super().__init__()
        dims = [in_dim] + [hidden] * (layers - 1) + [out_dim]
        self.lins = nn.ModuleList([nn.Linear(dims[i], dims[i + 1]) for i in range(layers)])
        self.dropout = dropout

    # edge_index принимаем, но не используем: так MLP вызывается так же, как GNN,
    # и подходит к той же функции train — сравнение получается честным
    def forward(self, x, edge_index=None, return_hidden=False):
        hidden = None
        for i, lin in enumerate(self.lins):
            x = lin(x)                       # каждая вершина обрабатывается сама по себе
            if i < len(self.lins) - 1:
                x = F.relu(x); hidden = x
                x = F.dropout(x, p=self.dropout, training=self.training)
        return (x, hidden) if return_hidden else x

In [ ]:
# TODO: допишите обучающий цикл с ранней остановкой по валидации
def train(model, epochs=200, lr=0.01, weight_decay=5e-4, patience=30, verbose=False):
    model = model.to(device)
    # weight_decay — L2-регуляризация: при 140 размеченных вершинах без неё сеть быстро переобучается
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val, best_state, waiting = 0.0, None, 0

    for epoch in range(epochs):
        model.train()                         # режим обучения: dropout включён
        opt.zero_grad()                       # градиенты в PyTorch накапливаются — обнуляем каждый шаг
        # ⚠ подаём ВЕСЬ граф, даже неразмеченные вершины: они нужны как соседи размеченных
        out = model(data.x, data.edge_index)  # [2708, 7] — логиты для всех вершин

        # TODO 1: посчитайте cross-entropy только по размеченным вершинам
        # подсказка: F.cross_entropy(out[маска], data.y[маска])
        # ⚠ какая маска? Метки val и test модель видеть не должна — иначе утечка
        loss = ...

        loss.backward(); opt.step()

        model.eval()                          # режим оценки: dropout выключен
        with torch.no_grad():                 # без графа вычислений — быстрее и меньше памяти
            pred = model(data.x, data.edge_index).argmax(1)   # класс с наибольшим логитом
            # TODO 2: доля верных предсказаний на валидации
            # подсказка: сравните pred и data.y на нужной маске, получится вектор из True/False
            val_acc = ...

        # ранняя остановка — уже написана: запоминаем веса с лучшей валидацией
        # и останавливаемся, если patience эпох подряд лучше не становится
        if val_acc > best_val:
            best_val, waiting = val_acc, 0
            # ⚠ clone обязателен: state_dict отдаёт ссылки на живые тензоры,
            # и без копии «лучшие» веса изменятся на следующем шаге оптимизатора
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            waiting += 1
            if waiting >= patience:
                break

    model.load_state_dict(best_state)         # возвращаем лучшие по валидации веса, а не последние
    model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(1)
        # тест считаем ОДИН раз в самом конце. Выбирать по тесту эпоху или гиперпараметры — утечка
        test_acc = float((pred[data.test_mask] == data.y[data.test_mask]).float().mean())
    return test_acc

acc = train(GNN(dataset.num_features, 64, dataset.num_classes))
print(f"GCN: test accuracy {acc:.3f}")

## 4. Обязательный baseline: а нужен ли граф вообще?

In [ ]:
def repeat(make_model, seeds=3, **kw):
    # make_model — функция, а не готовая модель: каждому сиду нужна свежая модель
    # со своей начальной инициализацией. (В домашнем задании сидов нужно не меньше пяти.)
    accs = []
    for seed in range(seeds):
        torch.manual_seed(seed)              # сид ставим ДО создания модели — от него зависят веса
        accs.append(train(make_model(), **kw))
    return float(np.mean(accs)), float(np.std(accs))

# три модели в одинаковых условиях: тот же train, те же гиперпараметры, тот же сплит
gcn_mean, gcn_std = repeat(lambda: GNN(dataset.num_features, 64, dataset.num_classes))
mlp_mean, mlp_std = repeat(lambda: MLP(dataset.num_features, 64, dataset.num_classes))
pyg_mean, pyg_std = repeat(lambda: GNN(dataset.num_features, 64, dataset.num_classes, conv=GCNConv))

print(f"наш GCN:      {gcn_mean:.3f} ± {gcn_std:.3f}")
print(f"GCNConv PyG:  {pyg_mean:.3f} ± {pyg_std:.3f}")
print(f"MLP без графа:{mlp_mean:.3f} ± {mlp_std:.3f}")

In [ ]:
plt.figure(figsize=(7, 3.2))
names = ["наш GCN", "GCNConv из PyG", "MLP без графа"]
values = [gcn_mean, pyg_mean, mlp_mean]
errors = [gcn_std, pyg_std, mlp_std]
# yerr — «усы»: стандартное отклонение по сидам. Разница меньше усов — это шум, а не победа
plt.bar(names, values, yerr=errors, capsize=5,
        color=["#1F3A5F", "#0E8F8F", "#6B7280"])
for i, v in enumerate(values):
    plt.text(i, v + 0.03, f"{v:.3f}", ha="center", fontweight="bold")
plt.ylim(0, 1.0); plt.ylabel("accuracy на тесте")
plt.title("Даёт ли граф выигрыш?"); plt.show()

Наша реализация должна совпасть с PyG в пределах разброса по сидам, а MLP — заметно отстать.
Если MLP не отстаёт, значит граф в этой задаче ничего не добавляет, и всю затею надо пересматривать.

## 5. Over-smoothing: делаем сеть глубже

In [ ]:
# TODO: посчитайте, насколько похожи представления вершин друг на друга
# нормируйте строки, посчитайте матрицу попарных косинусов и усредните вне диагонали
# ⚠ на диагонали стоит косинус вершины с самой собой — он всегда 1, его нужно исключить,
# иначе среднее окажется завышенным
def smoothness(hidden):
    ...

# проверка на случайной матрице: там сходство должно быть около нуля
print("случайные векторы:", round(smoothness(torch.randn(500, 32)), 3))

In [ ]:
depths, accs, sims = [2, 4, 8, 16], [], []
for L in depths:
    torch.manual_seed(0)                     # один и тот же сид — разница только в глубине
    model = GNN(dataset.num_features, 64, dataset.num_classes, layers=L)
    acc = train(model)
    model.eval()
    with torch.no_grad():
        # hidden — выход последнего скрытого слоя после ReLU: по нему меряем сходство вершин
        _, hidden = model(data.x, data.edge_index, return_hidden=True)
    accs.append(acc); sims.append(smoothness(hidden))
    print(f"{L:2d} слоёв: accuracy {acc:.3f}, сходство представлений {sims[-1]:.3f}")

# ⚠ два графика рядом, а не на одной оси: у accuracy и сходства разный смысл и масштаб
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(depths, accs, "o-", color="#1F3A5F"); ax[0].set_title("accuracy"); ax[0].set_xlabel("слоёв")
ax[1].plot(depths, sims, "o-", color="#E07A3F"); ax[1].set_title("сходство представлений"); ax[1].set_xlabel("слоёв")
plt.tight_layout(); plt.show()

Скорее всего, картина получилась не такой гладкой, как в лекции. Провал глубокой сети
бывает по двум разным причинам, и их полезно различать:

- **over-smoothing**: представления вершин слипаются, сходство стремится к единице
  (у 16 слоёв так и происходит);
- **сеть вообще не обучилась**: градиенты не доходят до первых слоёв, активации зануляются,
  и сходство оказывается близким к нулю просто потому, что все векторы нулевые.

Оба случая дают низкую accuracy, но лечатся по-разному. Посмотрите на своё значение
сходства и определите, что именно произошло у вас.

## 6. Лечим глубину residual-связями

In [ ]:
# TODO: добавьте residual-связь: выход слоя складывается со входом
# Осторожно: размерности должны совпадать, поэтому residual применяем только на скрытых слоях
class ResGNN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, layers=8, dropout=0.5):
        super().__init__()
        # вход проецируем в скрытую размерность ОДИН раз: h + f(h) требует одинаковых размеров
        self.inp = nn.Linear(in_dim, hidden)
        # layers - 1 скрытых слоёв hidden → hidden, плюс последний слой в классы
        self.convs = nn.ModuleList([MyGCNConv(hidden, hidden) for _ in range(layers - 1)])
        self.out = MyGCNConv(hidden, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index, return_hidden=False):
        # план:
        # 1. спроецируйте x в скрытую размерность через self.inp и примените ReLU;
        # 2. для каждого слоя из self.convs прибавьте к h его выход (после ReLU), затем dropout;
        # 3. последний слой self.out — без residual: у него другая выходная размерность;
        # 4. верните логиты, а при return_hidden=True — ещё и последний скрытый h
        ...

torch.manual_seed(0)
acc_res = train(ResGNN(dataset.num_features, 64, dataset.num_classes, layers=8))
print(f"8 слоёв с residual: {acc_res:.3f}   (без residual было {accs[depths.index(8)]:.3f})")

## 7. Выводы

- Слой GCN — это двадцать строк кода, и его обязательно нужно сверять с эталоном.
- Граф даёт большой прирост над MLP именно потому, что размеченных вершин мало.
- Глубина без специальных приёмов убивает модель: представления вершин слипаются.
- Residual-связи заметно смягчают проблему, но не делают глубокую сеть автоматически лучше.

### Задания для самостоятельной работы
1. Реализуйте `SAGEConv` (mean-агрегация плюс отдельная матрица для собственного вектора)
   и сравните с GCN.
2. Проверьте, что будет, если убрать петли из нормировки — насколько упадёт качество?
3. Постройте кривые обучения (loss и val accuracy по эпохам) для 2 и 8 слоёв —
   видно ли, что глубокая сеть не переобучается, а именно не учится?
4. Замените нормировку `1/sqrt(d_u d_v)` на `1/d_v` (усреднение). Изменится ли результат?